In [1]:
import pandas as pd

df = pd.read_csv(
    "../data/processed/PreProcessed_IBM_Telco_Customer_Churn_Cleaned.csv"
)

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,TenureGroup,TotalServices
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0-1 Year,1
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,No,One year,No,Mailed check,56.95,1889.50,No,2-4 Years,3
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,0-1 Year,3
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,2-4 Years,3
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,0-1 Year,1


In [2]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (7043, 23)

Columns:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn', 'TenureGroup', 'TotalServices']


In [3]:
print("Number of columns:", len(df.columns))

for i, col in enumerate(df.columns, start=1):
    print(i, col)

Number of columns: 23
1 customerID
2 gender
3 SeniorCitizen
4 Partner
5 Dependents
6 tenure
7 PhoneService
8 MultipleLines
9 InternetService
10 OnlineSecurity
11 OnlineBackup
12 DeviceProtection
13 TechSupport
14 StreamingTV
15 StreamingMovies
16 Contract
17 PaperlessBilling
18 PaymentMethod
19 MonthlyCharges
20 TotalCharges
21 Churn
22 TenureGroup
23 TotalServices


In [5]:
df["AverageMonthlySpend"] = (
    df["TotalCharges"] /
    df["tenure"].replace(0, 1)
)

AverageMonthlySpend: Created by dividing TotalCharges by tenure to estimate the customer's average monthly spending. This feature helps capture customer spending behavior for churn and LTV analysis.

In [6]:
df[[
    "tenure",
    "TotalCharges",
    "AverageMonthlySpend"
]].head(10)

,tenure,TotalCharges,AverageMonthlySpend
0,1,29.85,29.850000
1,34,1889.50,55.573529
2,2,108.15,54.075000
3,45,1840.75,40.905556
4,2,151.65,75.825000
5,8,820.50,102.562500
6,22,1949.40,88.609091
7,10,301.90,30.190000
8,28,3046.05,108.787500
9,62,3487.95,56.257258


In [7]:
df["AverageMonthlySpend"].describe()

count    7043.000000
mean       64.698218
std        30.270670
min         0.000000
25%        35.649000
50%        70.300000
75%        90.174158
max       121.400000
Name: AverageMonthlySpend, dtype: float64

In [8]:
df["ExpectedCharges"] = (
    df["MonthlyCharges"] * df["tenure"]
)

ExpectedCharges: Calculated by multiplying MonthlyCharges by tenure to estimate the customer's expected accumulated charges.

In [9]:
df[[
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "ExpectedCharges"
]].head(10)

,tenure,MonthlyCharges,TotalCharges,ExpectedCharges
0,1,29.85,29.85,29.85
1,34,56.95,1889.50,1936.30
2,2,53.85,108.15,107.70
3,45,42.30,1840.75,1903.50
4,2,70.70,151.65,141.40
5,8,99.65,820.50,797.20
6,22,89.10,1949.40,1960.20
7,10,29.75,301.90,297.50
8,28,104.80,3046.05,2934.40
9,62,56.15,3487.95,3481.30


In [10]:
df["ChargeDifference"] = (
    df["TotalCharges"] - df["ExpectedCharges"]
)

ChargeDifference: Calculated as TotalCharges - ExpectedCharges to capture the difference between actual and expected accumulated charges.

In [11]:
df[[
    "TotalCharges",
    "ExpectedCharges",
    "ChargeDifference"
]].head(10)

,TotalCharges,ExpectedCharges,ChargeDifference
0,29.85,29.85,0.00
1,1889.50,1936.30,-46.80
2,108.15,107.70,0.45
3,1840.75,1903.50,-62.75
4,151.65,141.40,10.25
5,820.50,797.20,23.30
6,1949.40,1960.20,-10.80
7,301.90,297.50,4.40
8,3046.05,2934.40,111.65
9,3487.95,3481.30,6.65


In [12]:
df["ServiceUsageRatio"] = df["TotalServices"] / 8

ServiceUsageRatio: Calculated as TotalServices / 8 to represent the proportion of available services subscribed to by each customer.

In [13]:
df[[
    "TotalServices",
    "ServiceUsageRatio"
]].head(10)

,TotalServices,ServiceUsageRatio
0,1,0.125
1,3,0.375
2,3,0.375
3,3,0.375
4,1,0.125
5,5,0.625
6,4,0.500
7,1,0.125
8,6,0.750
9,3,0.375


In [14]:
high_value_threshold = df["MonthlyCharges"].quantile(0.75)

df["HighValueCustomer"] = (
    df["MonthlyCharges"] >= high_value_threshold
).astype(int)

In [15]:
print("Threshold:", high_value_threshold)

print(
    df["HighValueCustomer"].value_counts()
)

Threshold: 89.85
HighValueCustomer
0    5272
1    1771
Name: count, dtype: int64


HighValueCustomer: Created as a binary feature identifying customers whose monthly charges are in the top 25% of customers.

In [16]:
df[[
    "MonthlyCharges",
    "HighValueCustomer"
]].head(10)

,MonthlyCharges,HighValueCustomer
0,29.85,0
1,56.95,0
2,53.85,0
3,42.30,0
4,70.70,0
5,99.65,1
6,89.10,0
7,29.75,0
8,104.80,1
9,56.15,0


In [17]:
df["ShortTenureCustomer"] = (
    df["tenure"] <= 12
).astype(int)

ShortTenureCustomer: Created as a binary feature to identify customers with a tenure of 12 months or less.

In [18]:
df[[
    "tenure",
    "ShortTenureCustomer"
]].head(10)

,tenure,ShortTenureCustomer
0,1,1
1,34,0
2,2,1
3,45,0
4,2,1
5,8,1
6,22,0
7,10,1
8,28,0
9,62,0


In [19]:
print(df["ShortTenureCustomer"].value_counts())

ShortTenureCustomer
0    4857
1    2186
Name: count, dtype: int64


In [20]:
new_features = [
    "AverageMonthlySpend",
    "ExpectedCharges",
    "ChargeDifference",
    "ServiceUsageRatio",
    "HighValueCustomer",
    "ShortTenureCustomer"
]

df[new_features].isnull().sum()

AverageMonthlySpend    0
ExpectedCharges        0
ChargeDifference       0
ServiceUsageRatio      0
HighValueCustomer      0
ShortTenureCustomer    0
dtype: int64

In [21]:
df[new_features].describe()

,AverageMonthlySpend,ExpectedCharges,ChargeDifference,ServiceUsageRatio,HighValueCustomer,ShortTenureCustomer
count,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000
mean,64.698218,2279.581350,0.152953,0.420364,0.251455,0.310379
std,30.270670,2264.729447,67.202778,0.257754,0.433880,0.462682
min,0.000000,0.000000,-370.850000,0.000000,0.000000,0.000000
25%,35.649000,394.000000,-28.600000,0.125000,0.000000,0.000000
50%,70.300000,1393.600000,0.000000,0.375000,0.000000,0.000000
75%,90.174158,3786.100000,28.500000,0.625000,1.000000,1.000000
max,121.400000,8550.000000,373.250000,1.000000,1.000000,1.000000


We need check this because we have done the Highvalue customer and ShortTenurecustomer values into 0 and 1's only. like Normalisation

In [22]:
print(df["HighValueCustomer"].unique())
print(df["ShortTenureCustomer"].unique())

[0 1]
[1 0]


In [23]:
print("Number of columns:", len(df.columns))

for i, col in enumerate(df.columns, start=1):
    print(i, col)

Number of columns: 29
1 customerID
2 gender
3 SeniorCitizen
4 Partner
5 Dependents
6 tenure
7 PhoneService
8 MultipleLines
9 InternetService
10 OnlineSecurity
11 OnlineBackup
12 DeviceProtection
13 TechSupport
14 StreamingTV
15 StreamingMovies
16 Contract
17 PaperlessBilling
18 PaymentMethod
19 MonthlyCharges
20 TotalCharges
21 Churn
22 TenureGroup
23 TotalServices
24 AverageMonthlySpend
25 ExpectedCharges
26 ChargeDifference
27 ServiceUsageRatio
28 HighValueCustomer
29 ShortTenureCustomer


In [24]:
numeric_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "TotalServices",
    "AverageMonthlySpend",
    "ExpectedCharges",
    "ChargeDifference",
    "ServiceUsageRatio",
    "HighValueCustomer",
    "ShortTenureCustomer"
]

df[numeric_features].corr()

,tenure,MonthlyCharges,TotalCharges,TotalServices,AverageMonthlySpend,ExpectedCharges,ChargeDifference,ServiceUsageRatio,HighValueCustomer,ShortTenureCustomer
tenure,1.000000,0.247900,0.826178,0.523600,0.249391,0.826568,0.012249,0.523600,0.291685,-0.755067
MonthlyCharges,0.247900,1.000000,0.651174,0.802322,0.994355,0.651566,0.006781,0.802322,0.694559,-0.193180
TotalCharges,0.826178,0.651174,1.000000,0.795854,0.651435,0.999561,0.045538,0.795854,0.588165,-0.593290
TotalServices,0.523600,0.802322,0.795854,1.000000,0.797374,0.796061,0.017473,1.000000,0.622872,-0.398202
AverageMonthlySpend,0.249391,0.994355,0.651435,0.797374,1.000000,0.650134,0.063853,0.797374,0.692107,-0.194162
ExpectedCharges,0.826568,0.651566,0.999561,0.796061,0.650134,1.000000,0.015905,0.796061,0.587823,-0.593863
ChargeDifference,0.012249,0.006781,0.045538,0.017473,0.063853,0.015905,1.000000,0.017473,0.029605,0.001106
ServiceUsageRatio,0.523600,0.802322,0.795854,1.000000,0.797374,0.796061,0.017473,1.000000,0.622872,-0.398202
HighValueCustomer,0.291685,0.694559,0.588165,0.622872,0.692107,0.587823,0.029605,0.622872,1.000000,-0.231795
ShortTenureCustomer,-0.755067,-0.193180,-0.593290,-0.398202,-0.194162,-0.593863,0.001106,-0.398202,-0.231795,1.000000


In [25]:
import os

os.makedirs("../data/processed", exist_ok=True)

output_path = "../data/processed/Feature_Engineered_Telco_Customer_Churn.csv"

df.to_csv(output_path, index=False)

print("Dataset saved successfully!")
print("Path:", os.path.abspath(output_path))

Dataset saved successfully!
Path: c:\Users\saisr\Customer-Churn-Prediction-And-Lifetime-Value-LTV-Engine\data\processed\Feature_Engineered_Telco_Customer_Churn.csv
